In [1]:
"""
SmartRetail360 — Complete Pipeline
====================================
Run every cell in Jupyter Notebook (Shift+Enter each cell).
Wait for [number] before running the next cell.
Dataset: Anonymized_Restaurant_Sales_Data.csv
"""

# ══════════════════════════════════════════════════════════════
# CELL 1 — Setup paths and imports
# ══════════════════════════════════════════════════════════════

import os, sys, warnings, sqlite3, pickle, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import classification_report, roc_auc_score, mean_absolute_error

warnings.filterwarnings("ignore")

PROJECT   = "/Users/komudigimhani/Desktop/SmartRetail360"
DATA_FILE = os.path.join(PROJECT, "data", "raw", "Anonymized_Restaurant_Sales_Data.csv")
DB_PATH   = os.path.join(PROJECT, "data", "smartretail.db")

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
os.makedirs(os.path.join(PROJECT, "data", "processed"), exist_ok=True)
os.makedirs(os.path.join(PROJECT, "models"),  exist_ok=True)
os.makedirs(os.path.join(PROJECT, "reports"), exist_ok=True)
os.makedirs(os.path.join(PROJECT, "app"),     exist_ok=True)

print("✅ Setup complete!")
print(f"   Project  : {PROJECT}")
print(f"   Dataset  : {DATA_FILE}")
print(f"   DB path  : {DB_PATH}")


# ══════════════════════════════════════════════════════════════
# CELL 2 — Load raw data
# ══════════════════════════════════════════════════════════════

print("Loading dataset...")
df_raw = pd.read_csv(DATA_FILE)

print(f"✅ Loaded {len(df_raw):,} rows")
print(f"   Columns : {list(df_raw.columns)}")
print(df_raw.head(3).to_string())


# ══════════════════════════════════════════════════════════════
# CELL 3 — Clean data
# ══════════════════════════════════════════════════════════════

df = df_raw.copy()

# Remove cancelled orders
df = df[df["Cancelled"] == "No"].copy()

# Parse date and time
df["InvoiceDate"] = pd.to_datetime(df["Date"] + " " + df["Time"],
                                    format="%d/%m/%Y %H:%M", errors="coerce")
df = df.dropna(subset=["InvoiceDate"])

# Clean OrderID — strip prefix and convert
df["OrderID"] = df["OrderID"].astype(str).str.replace("RES_ORD_", "", regex=False).str.replace(".0", "", regex=False).str.strip()

# Rename for clarity
df = df.rename(columns={
    "Line item name" : "Description",
    "Price Per Item" : "Price",
    "Gross Sales"    : "line_revenue",
    "Est. Profit"    : "profit",
    "Est. Cost"      : "cost",
})

# Drop original Date and Time columns to avoid duplicate column name error
df = df.drop(columns=["Date", "Time"], errors="ignore")

# Time features
df["date"]        = df["InvoiceDate"].dt.date
df["year"]        = df["InvoiceDate"].dt.year
df["month"]       = df["InvoiceDate"].dt.month
df["day_of_week"] = df["InvoiceDate"].dt.dayofweek   # 0=Mon 6=Sun
df["hour"]        = df["InvoiceDate"].dt.hour

df = df.reset_index(drop=True)

print(f"✅ Clean data: {len(df):,} rows  (removed cancelled orders)")
print(f"   Orders    : {df['OrderID'].nunique():,}")
print(f"   Products  : {df['Description'].nunique():,}")
print(f"   Categories: {df['Category'].nunique()}")
print(f"   Revenue   : £{df['line_revenue'].sum():,.0f}")
print(f"   Profit    : £{df['profit'].sum():,.0f}")
print(f"   Dates     : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")


# ══════════════════════════════════════════════════════════════
# CELL 4 — Save to SQLite database
# ══════════════════════════════════════════════════════════════

conn = sqlite3.connect(DB_PATH)
df.to_sql("transactions", conn, if_exists="replace", index=False)
conn.close()

print(f"✅ Saved transactions to SQLite: {DB_PATH}")


# ══════════════════════════════════════════════════════════════
# CELL 5 — RFM Analysis (using OrderID as "Customer")
# ══════════════════════════════════════════════════════════════
# NOTE: This dataset has no customer IDs.
# We use OrderID-level RFM to analyse order behaviour patterns.
# Recency  = days since last order date for that order group
# Frequency = how many times an order appeared (items per order)
# Monetary  = total spend per order

snapshot = df["InvoiceDate"].max() + timedelta(days=1)

rfm = df.groupby("OrderID").agg(
    Recency   = ("InvoiceDate",  lambda x: (snapshot - x.max()).days),
    Frequency = ("Description",  "count"),        # items in order
    Monetary  = ("line_revenue", "sum"),           # order total
    OrderDate = ("InvoiceDate",  "max"),
).reset_index()

# RFM Scores 1-5
rfm["R_Score"] = pd.qcut(rfm["Recency"],   5, labels=[5,4,3,2,1], duplicates="drop").astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"),  5, labels=[1,2,3,4,5]).astype(int)
rfm["RFM_Total"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

# Segments
def assign_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    total = r + f + m
    if total >= 12:             return "High Value"
    elif total >= 9:            return "Good Value"
    elif total >= 6:            return "Average"
    elif r <= 2:                return "Lapsed"
    else:                       return "Low Value"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

print(f"✅ RFM built for {len(rfm):,} orders")
print("\nSegment distribution:")
print(rfm["Segment"].value_counts().to_string())


# ══════════════════════════════════════════════════════════════
# CELL 6 — Customer Segmentation (KMeans on RFM)
# ══════════════════════════════════════════════════════════════

features_for_cluster = rfm[["Recency", "Frequency", "Monetary"]].copy()
features_for_cluster["Frequency"] = np.log1p(features_for_cluster["Frequency"])
features_for_cluster["Monetary"]  = np.log1p(features_for_cluster["Monetary"])

scaler_rfm = StandardScaler()
X_rfm = scaler_rfm.fit_transform(features_for_cluster)

# Find best K 2-6
sil_scores = []
k_range    = range(2, 7)
for k in k_range:
    km_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels  = km_test.fit_predict(X_rfm)
    sil_scores.append(silhouette_score(X_rfm, labels))

best_k = list(k_range)[np.argmax(sil_scores)]
best_k = max(3, min(best_k, 5))   # keep 3-5

km = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm["Cluster"] = km.fit_predict(X_rfm)

final_sil = silhouette_score(X_rfm, rfm["Cluster"])
print(f"✅ KMeans done: K={best_k}, Silhouette={final_sil:.3f}")
print(rfm["Cluster"].value_counts().sort_index().to_string())

with open(os.path.join(PROJECT, "models", "kmeans_model.pkl"), "wb") as f:
    pickle.dump(km, f)
with open(os.path.join(PROJECT, "models", "rfm_scaler.pkl"), "wb") as f:
    pickle.dump(scaler_rfm, f)

print("✅ KMeans model saved!")


# ══════════════════════════════════════════════════════════════
# CELL 7 — Revenue Trend Forecasting (GBM)
# ══════════════════════════════════════════════════════════════

daily = df.groupby("date").agg(
    revenue  = ("line_revenue", "sum"),
    profit   = ("profit",       "sum"),
    quantity = ("Quantity",     "sum"),
    n_orders = ("OrderID",      "nunique"),
).reset_index()
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").reset_index(drop=True)

daily["day_of_week"] = daily["date"].dt.dayofweek
daily["month"]       = daily["date"].dt.month
daily["is_weekend"]  = (daily["day_of_week"] >= 5).astype(int)
daily["trend"]       = np.arange(len(daily))

for lag in [1, 7, 14, 28]:
    daily[f"lag_{lag}"] = daily["revenue"].shift(lag)
daily["rolling_7"]  = daily["revenue"].shift(1).rolling(7).mean()
daily["rolling_28"] = daily["revenue"].shift(1).rolling(28).mean()

daily_clean = daily.dropna().reset_index(drop=True)

feat_cols_ts = ["day_of_week", "month", "is_weekend", "trend",
                "lag_1", "lag_7", "lag_14", "lag_28",
                "rolling_7", "rolling_28"]

split      = int(len(daily_clean) * 0.85)
train_ts   = daily_clean.iloc[:split]
test_ts    = daily_clean.iloc[split:]

forecast_model = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
forecast_model.fit(train_ts[feat_cols_ts], train_ts["revenue"])

preds = forecast_model.predict(test_ts[feat_cols_ts])
preds = np.maximum(preds, 0)

mae  = mean_absolute_error(test_ts["revenue"], preds)
mape = np.mean(np.abs((test_ts["revenue"].values - preds) / (test_ts["revenue"].values + 1))) * 100
print(f"✅ Forecast model — MAE: £{mae:,.0f} | MAPE: {mape:.1f}%")

with open(os.path.join(PROJECT, "models", "forecast_model.pkl"), "wb") as f:
    pickle.dump(forecast_model, f)
with open(os.path.join(PROJECT, "models", "forecast_features.json"), "w") as f:
    json.dump(feat_cols_ts, f)
with open(os.path.join(PROJECT, "models", "forecast_meta.json"), "w") as f:
    json.dump({"mae": round(mae, 2), "mape": round(mape, 2)}, f)

daily.to_csv(os.path.join(PROJECT, "data", "processed", "daily_timeseries.csv"), index=False)
print("✅ Forecast model saved!")


# ══════════════════════════════════════════════════════════════
# CELL 8 — Cancellation Prediction (Random Forest)
# ══════════════════════════════════════════════════════════════
# Using full raw data (including cancelled) to train model

df_all = df_raw.copy()
df_all["InvoiceDate"] = pd.to_datetime(
    df_all["Date"] + " " + df_all["Time"], format="%d/%m/%Y %H:%M", errors="coerce")
df_all = df_all.dropna(subset=["InvoiceDate"])
df_all["hour"]       = df_all["InvoiceDate"].dt.hour
df_all["day_of_week"]= df_all["InvoiceDate"].dt.dayofweek
df_all["month"]      = df_all["InvoiceDate"].dt.month
df_all["is_weekend"] = (df_all["day_of_week"] >= 5).astype(int)
df_all["is_cancelled"]= (df_all["Cancelled"] == "Yes").astype(int)

# Encode categoricals
df_all["OrderType_enc"] = (df_all["OrderType"] == "Delivery").astype(int)
cat_dummies = pd.get_dummies(df_all["Category"], prefix="cat")
df_all = pd.concat([df_all, cat_dummies], axis=1)

cat_cols = [c for c in df_all.columns if c.startswith("cat_")]
feat_cols_cancel = ["hour", "day_of_week", "month", "is_weekend",
                     "OrderType_enc", "Quantity", "Price Per Item"] + cat_cols

X = df_all[feat_cols_cancel].fillna(0)
y = df_all["is_cancelled"]

cancel_rate = y.mean()
print(f"Cancellation rate: {cancel_rate:.1%}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

scaler_cancel = StandardScaler()
X_tr_s = scaler_cancel.fit_transform(X_train)
X_te_s = scaler_cancel.transform(X_test)

cancel_model = RandomForestClassifier(
    n_estimators=200, max_depth=6, random_state=42,
    class_weight="balanced", n_jobs=-1)
cancel_model.fit(X_tr_s, y_train)

y_proba = cancel_model.predict_proba(X_te_s)[:, 1]
try:
    auc = roc_auc_score(y_test, y_proba)
    print(f"✅ Cancellation model AUC-ROC = {auc:.4f}")
except Exception as e:
    auc = 0.5
    print(f"AUC note: {e}")

print(classification_report(y_test, cancel_model.predict(X_te_s),
                             target_names=["Active", "Cancelled"]))

with open(os.path.join(PROJECT, "models", "cancel_model.pkl"), "wb") as f:
    pickle.dump(cancel_model, f)
with open(os.path.join(PROJECT, "models", "cancel_scaler.pkl"), "wb") as f:
    pickle.dump(scaler_cancel, f)
with open(os.path.join(PROJECT, "models", "cancel_features.json"), "w") as f:
    json.dump(feat_cols_cancel, f)
with open(os.path.join(PROJECT, "models", "cancel_meta.json"), "w") as f:
    json.dump({"auc": round(auc, 4), "cancel_rate": round(float(cancel_rate), 4)}, f)

print("✅ Cancellation model saved!")


# ══════════════════════════════════════════════════════════════
# CELL 9 — Sentiment Analysis on Menu Items
# ══════════════════════════════════════════════════════════════

import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Neutral"
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:    return "Positive"
    elif score <= -0.05: return "Negative"
    else:                return "Neutral"

products_sentiment = df.groupby(["Description", "Category"]).agg(
    total_revenue   = ("line_revenue", "sum"),
    total_sold      = ("Quantity",     "sum"),
    n_orders        = ("OrderID",      "nunique"),
    avg_price       = ("Price",        "mean"),
    total_profit    = ("profit",       "sum"),
).reset_index()

products_sentiment["Sentiment"] = products_sentiment["Description"].apply(get_sentiment)
products_sentiment["SentimentScore"] = products_sentiment["Description"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"] if pd.notna(x) else 0)

print(f"✅ Sentiment analysed for {len(products_sentiment):,} menu items")
print(products_sentiment["Sentiment"].value_counts().to_string())

products_sentiment.to_csv(
    os.path.join(PROJECT, "data", "processed", "product_sentiment.csv"), index=False)


# ══════════════════════════════════════════════════════════════
# CELL 10 — Save all data to SQLite
# ══════════════════════════════════════════════════════════════

conn = sqlite3.connect(DB_PATH)

df.to_sql("transactions", conn, if_exists="replace", index=False)
rfm.to_sql("rfm", conn, if_exists="replace", index=False)
daily.to_sql("daily_timeseries", conn, if_exists="replace", index=False)

products_table = df.groupby(["Description", "Category"]).agg(
    total_revenue = ("line_revenue", "sum"),
    total_qty     = ("Quantity",     "sum"),
    n_orders      = ("OrderID",      "nunique"),
    avg_price     = ("Price",        "mean"),
    total_profit  = ("profit",       "sum"),
).reset_index()
products_table.to_sql("products", conn, if_exists="replace", index=False)

category_rev = df.groupby("Category").agg(
    revenue  = ("line_revenue", "sum"),
    profit   = ("profit",       "sum"),
    orders   = ("OrderID",      "nunique"),
    qty      = ("Quantity",     "sum"),
).reset_index()
category_rev.to_sql("category_revenue", conn, if_exists="replace", index=False)

ordertype_rev = df.groupby("OrderType").agg(
    revenue  = ("line_revenue", "sum"),
    profit   = ("profit",       "sum"),
    orders   = ("OrderID",      "nunique"),
).reset_index()
ordertype_rev.to_sql("ordertype_revenue", conn, if_exists="replace", index=False)

products_sentiment.to_sql("product_sentiment", conn, if_exists="replace", index=False)

conn.close()
print(f"✅ All data saved to SQLite: {DB_PATH}")


# ══════════════════════════════════════════════════════════════
# CELL 11 — Save EDA charts
# ══════════════════════════════════════════════════════════════

reports_dir = os.path.join(PROJECT, "reports")

# Monthly revenue
monthly = df.groupby(["year", "month"])["line_revenue"].sum().reset_index()
monthly["period"] = monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly["period"], monthly["line_revenue"], color="#E87040", alpha=0.85)
ax.set_title("Monthly Revenue (£)", fontsize=13)
ax.set_xlabel("Month"); ax.set_ylabel("Revenue (£)")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, "monthly_revenue.png"), dpi=120)
plt.close()

# Top 10 menu items
top10 = products_table.nlargest(10, "total_revenue")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top10["Description"].str[:40], top10["total_revenue"], color="#378ADD", alpha=0.85)
ax.set_title("Top 10 Menu Items by Revenue (£)", fontsize=13)
ax.set_xlabel("Revenue (£)"); ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, "top_products.png"), dpi=120)
plt.close()

# Category distribution
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#E87040","#378ADD","#1D9E75","#F0C27F","#9B59B6","#E24B4A","#34495E","#F39C12"]
ax.bar(category_rev["Category"], category_rev["revenue"],
       color=colors[:len(category_rev)], alpha=0.85)
ax.set_title("Revenue by Category (£)", fontsize=13)
ax.set_ylabel("Revenue (£)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, "category_revenue.png"), dpi=120)
plt.close()

print("✅ EDA charts saved to reports/")


# ══════════════════════════════════════════════════════════════
# CELL 12 — Final summary
# ══════════════════════════════════════════════════════════════

print("\n" + "═"*55)
print("  ✅  SMARTRETAIL360 PIPELINE COMPLETE!")
print("═"*55)

files_to_check = [
    ("models/kmeans_model.pkl",                  "KMeans Segmentation"),
    ("models/forecast_model.pkl",                "Revenue Forecast"),
    ("models/cancel_model.pkl",                  "Cancellation Predictor"),
    ("data/processed/daily_timeseries.csv",      "Time Series Data"),
    ("data/processed/product_sentiment.csv",     "Sentiment Data"),
    ("data/smartretail.db",                      "SQLite Database"),
]

all_ok = True
for fpath, label in files_to_check:
    full   = os.path.join(PROJECT, fpath)
    exists = os.path.exists(full)
    status = "✅" if exists else "❌"
    print(f"  {status} {label}")
    if not exists: all_ok = False

if all_ok:
    print("\n  ✅ ALL FILES READY!")
    print("\n  ── HOW TO LAUNCH DASHBOARD ──")
    print("  1. Open a NEW Terminal window")
    print(f"  2. cd {PROJECT}")
    print("  3. streamlit run app/Home.py")
    print("  4. Browser opens → http://localhost:8501")
else:
    print("\n  ❌ Some files missing. Re-run cells above.")


✅ Setup complete!
   Project  : /Users/komudigimhani/Desktop/SmartRetail360
   Dataset  : /Users/komudigimhani/Desktop/SmartRetail360/data/raw/Anonymized_Restaurant_Sales_Data.csv
   DB path  : /Users/komudigimhani/Desktop/SmartRetail360/data/smartretail.db
Loading dataset...
✅ Loaded 4,385 rows
   Columns : ['OrderID', 'Date', 'DayOfWeek', 'Time', 'Category', 'Line item name', 'Quantity', 'Price Per Item', 'Gross Sales', 'Est. Cost', 'Est. Profit', 'OrderType', 'Payment', 'Cancelled']
         OrderID        Date DayOfWeek   Time      Category                            Line item name  Quantity  Price Per Item  Gross Sales  Est. Cost  Est. Profit OrderType Payment Cancelled
0  RES_ORD_931.0  03/01/2023   Tuesday  18:50     APETISERS                              Chicken Suya         2             3.5          7.0       2.45         4.55  Delivery    Card        No
1  RES_ORD_930.0  03/01/2023   Tuesday  18:32  MAIN COURSES  Boiled Rice with Designer Stew - Ayamase         1            